# BlitzMart Analytics — Notebook 2: SQL Analysis

This notebook answers 15 core business questions about BlitzMart using SQL, run against the cleaned dataset produced in Notebook 1.

**Why DuckDB:** DuckDB runs PostgreSQL-compatible SQL directly on CSV files with no database server to set up, and it handles the 3.4M-row `order_items` table in seconds. The SQL skills here — CTEs, window functions, RFM segmentation, cohort analysis — transfer directly to PostgreSQL, BigQuery, or Snowflake.

**Structure:** The 15 queries are grouped into four themes — Revenue & Sales, Customer Analytics, Operations & Supply Chain, and Advanced Analytics. Each query sits in its own cell with the business question it answers explained directly above it.

The final section exports key query results as CSV files for the Tableau dashboards.

---

## 1. Setup

### 1.1 Connect to Google Drive

The cleaned dataset from Notebook 1 lives in Google Drive. Mounting Drive gives this notebook read access to those `data_clean/` CSV files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Install DuckDB

DuckDB is the SQL engine for this analysis. Faker is included here only because the data-refresh cell in the next section uses it. Both install in a few seconds.

In [ ]:
!pip install duckdb faker -q
print("DuckDB + Faker installed")

## 2. Data Refresh — Realistic Carrier & Return Patterns

The data generated in Notebook 1 used uniform random distributions for shipments and returns, which made every carrier and every product category perform almost identically. That isn't how a real business looks — some carriers genuinely outperform others, and some categories are returned far more often than others.

This cell refreshes the `shipments` and `returns` tables with realistic, research-informed patterns:

- **Carrier performance gaps.** Each carrier gets its own on-time rate and delay profile, modelled on the real German parcel market — DHL as the premium reliable option (~82% on-time), GLS as the budget option (~48%), with DPD, UPS, and Hermes in between.
- **Warehouse efficiency.** Each warehouse applies a speed multiplier — urban hubs like Berlin are faster, peripheral hubs like Leipzig are slower — so fulfilment performance varies by location.
- **Category-specific return rates.** Fashion has the highest return rate (try-on culture), Books the lowest (final-sale mindset), matching published e-commerce benchmarks.
- **Return reasons mapped to categories.** "Wrong size" concentrates in Fashion, "Damaged in transit" in Electronics — the reason fits the product type.

This cell only needs to run **once** per environment. After it has refreshed the files in Drive, it can be skipped on later sessions.

In [ ]:
"""Refresh shipments & returns with realistic business patterns."""

import os
import numpy as np
import pandas as pd

np.random.seed(42)

DATA_PATH = "/content/drive/MyDrive/BlitzMart_Analytics/data_clean"

orders      = pd.read_csv(f"{DATA_PATH}/orders.csv", parse_dates=["order_date"])
order_items = pd.read_csv(f"{DATA_PATH}/order_items.csv")
products    = pd.read_csv(f"{DATA_PATH}/products.csv")

# ---------- 1. SHIPMENTS — realistic carrier gaps ----------
# Each carrier has its own profile: (market_share, on_time_rate,
# avg_delay_when_late). Values are modelled on the real German parcel
# market — DHL premium and reliable, GLS budget and slower.
print("Rebuilding shipments with realistic carrier performance...")

carrier_profile = {
    "DHL":    (0.50, 0.82, 1.2),
    "DPD":    (0.15, 0.72, 1.5),
    "UPS":    (0.08, 0.75, 1.4),
    "Hermes": (0.20, 0.58, 2.3),
    "GLS":    (0.07, 0.48, 2.8),
}
# Warehouse speed multiplier — urban hubs faster, peripheral hubs slower.
warehouse_efficiency = {1: 0.95, 2: 0.97, 3: 1.00, 4: 1.05, 5: 1.10, 6: 1.20}

shipped_mask = orders["order_status"].isin(["Delivered", "Shipped", "Returned"])
shipped = orders.loc[shipped_mask, ["order_id", "order_date", "warehouse_id"]].reset_index(drop=True)
n = len(shipped)

# Assign carriers by market share.
carriers_arr = np.array(list(carrier_profile.keys()))
shares = np.array([v[0] for v in carrier_profile.values()]); shares /= shares.sum()
carriers = np.random.choice(carriers_arr, size=n, p=shares)

# Actual delivery time depends on the carrier's on-time probability and
# the warehouse efficiency multiplier. On-time shipments arrive at or
# before the promised day; late ones add a carrier-specific delay.
promised = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.20, 0.50, 0.20, 0.05])
on_time_probs = np.array([carrier_profile[c][1] for c in carriers])
avg_delays    = np.array([carrier_profile[c][2] for c in carriers])
wh_mult       = shipped["warehouse_id"].map(warehouse_efficiency).to_numpy()

is_on_time = np.random.random(n) < on_time_probs
actual = np.where(
    is_on_time,
    np.maximum(1, (promised - np.random.randint(0, 2, size=n)) * wh_mult).round().astype(int),
    (promised + np.maximum(1, np.random.poisson(avg_delays, size=n)) * wh_mult).round().astype(int)
)
actual = np.clip(actual, 1, 14)
on_time_flag = (actual <= promised).astype(int)

ship_dates = pd.to_datetime(shipped["order_date"]) + pd.to_timedelta(np.random.randint(0, 2, size=n), unit="D")
delivery_dates = ship_dates + pd.to_timedelta(actual, unit="D")

shipments_new = pd.DataFrame({
    "shipment_id": np.arange(1, n + 1),
    "order_id": shipped["order_id"].values,
    "carrier": carriers,
    "ship_date": ship_dates,
    "promised_delivery_days": promised,
    "actual_delivery_days": actual,
    "delivery_date": delivery_dates,
    "on_time_delivery": on_time_flag,
})
shipments_new.to_csv(f"{DATA_PATH}/shipments.csv", index=False)
print(f"  {n:,} shipments rebuilt")

# ---------- 2. RETURNS — category-specific rates ----------
# Each category gets its own return rate, return-reason distribution, and
# refund range, all based on published e-commerce benchmarks.
print("\nRebuilding returns with category-specific patterns...")

category_return_rates = {
    "Fashion": 0.22, "Toys": 0.12, "Sports & Outdoors": 0.10, "Electronics": 0.08,
    "Home & Kitchen": 0.06, "Beauty": 0.05, "Automotive": 0.04, "Books": 0.03,
}
category_reason_weights = {
    "Fashion":     {"Wrong size":0.55,"Not as described":0.20,"Changed mind":0.15,"Quality issue":0.07,"Damaged in transit":0.02,"Wrong item shipped":0.01},
    "Electronics": {"Damaged in transit":0.35,"Quality issue":0.25,"Not as described":0.15,"Changed mind":0.15,"Wrong item shipped":0.08,"Wrong size":0.02},
    "Home & Kitchen": {"Damaged in transit":0.30,"Not as described":0.25,"Quality issue":0.20,"Changed mind":0.18,"Wrong item shipped":0.05,"Wrong size":0.02},
    "Beauty":      {"Quality issue":0.30,"Not as described":0.25,"Changed mind":0.30,"Damaged in transit":0.10,"Wrong item shipped":0.04,"Wrong size":0.01},
    "Sports & Outdoors": {"Wrong size":0.30,"Quality issue":0.25,"Not as described":0.20,"Changed mind":0.15,"Damaged in transit":0.08,"Wrong item shipped":0.02},
    "Books":       {"Damaged in transit":0.40,"Changed mind":0.30,"Not as described":0.20,"Wrong item shipped":0.07,"Quality issue":0.02,"Wrong size":0.01},
    "Toys":        {"Quality issue":0.30,"Damaged in transit":0.25,"Not as described":0.20,"Changed mind":0.15,"Wrong item shipped":0.07,"Wrong size":0.03},
    "Automotive":  {"Not as described":0.30,"Quality issue":0.25,"Wrong item shipped":0.20,"Damaged in transit":0.15,"Changed mind":0.08,"Wrong size":0.02},
}
category_refund_range = {
    "Fashion": (30, 200), "Electronics": (100, 1500), "Home & Kitchen": (40, 400),
    "Beauty": (15, 100), "Sports & Outdoors": (50, 600), "Books": (10, 50),
    "Toys": (20, 200), "Automotive": (50, 800),
}

# Each order's category is taken from its first line item — that drives
# which return rate and reason distribution applies.
product_cat = products.set_index("product_id")["category"].to_dict()
first_item = order_items.drop_duplicates("order_id", keep="first")[["order_id", "product_id"]].copy()
first_item["category"] = first_item["product_id"].map(product_cat)
order_to_cat = dict(zip(first_item["order_id"], first_item["category"]))

eligible = orders[orders["order_status"].isin(["Delivered", "Returned"])][["order_id", "order_date"]].copy()
eligible["category"] = eligible["order_id"].map(order_to_cat)
eligible = eligible.dropna(subset=["category"])
eligible["return_prob"] = eligible["category"].map(category_return_rates)
eligible["is_returned"] = np.random.random(len(eligible)) < eligible["return_prob"]

returned = eligible[eligible["is_returned"]].reset_index(drop=True)
n_ret = len(returned)

reasons = []
refunds = []
for cat in returned["category"]:
    w = category_reason_weights[cat]
    probs = np.array(list(w.values())); probs /= probs.sum()
    reasons.append(np.random.choice(list(w.keys()), p=probs))
    lo, hi = category_refund_range[cat]
    refunds.append(round(np.random.uniform(lo, hi), 2))

return_dates = pd.to_datetime(returned["order_date"]) + pd.to_timedelta(np.random.randint(3, 30, size=n_ret), unit="D")

returns_new = pd.DataFrame({
    "return_id": np.arange(1, n_ret + 1),
    "order_id": returned["order_id"].values,
    "return_date": return_dates.values,
    "return_reason": reasons,
    "refund_amount": refunds,
})
returns_new.to_csv(f"{DATA_PATH}/returns.csv", index=False)

# Keep order_status consistent with the new returns table: any order now
# in returns is marked "Returned", and any previously-returned order no
# longer in returns is flipped back to "Delivered".
orders.loc[orders["order_id"].isin(returns_new["order_id"]), "order_status"] = "Returned"
flip_to_delivered = ~orders["order_id"].isin(returns_new["order_id"]) & (orders["order_status"] == "Returned")
orders.loc[flip_to_delivered, "order_status"] = "Delivered"
orders.to_csv(f"{DATA_PATH}/orders.csv", index=False)

print(f"  {n_ret:,} returns rebuilt")
print("\n=== REFRESH COMPLETE ===")
print("\nNew carrier on-time rates:")
for c in ["DHL", "DPD", "UPS", "Hermes", "GLS"]:
    rate = shipments_new[shipments_new["carrier"]==c]["on_time_delivery"].mean() * 100
    print(f"  {c:<8} {rate:5.1f}%")

## 3. Load Data into DuckDB

This cell creates an in-memory DuckDB database and loads all 8 cleaned tables into it. An in-memory database is fast and disposable — it rebuilds from the CSV files every session, which is fine because the CSVs in Drive are the permanent source of truth.

Loading the tables as proper DuckDB tables (rather than querying the CSVs directly each time) makes the joins in the following queries noticeably faster.

In [ ]:
import duckdb

CLEAN_PATH = "/content/drive/MyDrive/BlitzMart_Analytics/data_clean"
con = duckdb.connect(database=":memory:")

tables = ["warehouses", "suppliers", "products", "customers",
          "orders", "order_items", "shipments", "returns"]

for t in tables:
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('{CLEAN_PATH}/{t}.csv')")
    count = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<15} {count:>10,} rows")

print("\nAll tables loaded")

## 4. SQL Analysis

The 15 queries below are grouped into four themes. Every query answers a specific business question, stated directly above it.

### 4.1 Revenue & Sales

The first five queries establish the commercial picture: how much revenue the business makes, how it trends over time, and where it comes from by category, product, and warehouse.

**Query 1 — Topline KPIs**

The single-row health snapshot of the business: total orders, unique customers, total revenue, average order value (AOV), and average items per order. This is the foundation every other query builds on. The `WHERE` clause restricts to `Delivered` and `Shipped` orders because cancelled and processing orders haven't generated realised revenue.

In [ ]:
con.execute("""
SELECT
    COUNT(DISTINCT o.order_id)                                AS total_orders,
    COUNT(DISTINCT o.customer_id)                             AS unique_customers,
    ROUND(SUM(oi.line_total), 0)                              AS total_revenue_eur,
    ROUND(SUM(oi.line_total) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value_eur,
    ROUND(SUM(oi.quantity)::DECIMAL
          / COUNT(DISTINCT o.order_id), 2)                    AS avg_items_per_order
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status IN ('Delivered', 'Shipped')
""").fetchdf()

**Query 2 — Monthly Revenue & Year-over-Year Growth**

Revenue aggregated by month, with a `LAG(revenue, 12)` window function pulling the same month from the previous year to calculate YoY growth %. This exposes both the seasonal pattern (the November–December spike) and whether the business is growing year over year. `NULLIF` guards against divide-by-zero in the first 12 months, which have no prior-year comparison.

In [ ]:
con.execute("""
WITH monthly AS (
    SELECT DATE_TRUNC('month', o.order_date) AS month,
           SUM(oi.line_total)                AS revenue,
           COUNT(DISTINCT o.order_id)        AS orders
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status IN ('Delivered', 'Shipped')
    GROUP BY 1
)
SELECT month, orders,
       ROUND(revenue, 0)                                AS revenue_eur,
       ROUND(LAG(revenue, 12) OVER (ORDER BY month), 0) AS revenue_last_year,
       ROUND(100.0 * (revenue - LAG(revenue, 12) OVER (ORDER BY month))
             / NULLIF(LAG(revenue, 12) OVER (ORDER BY month), 0), 1) AS yoy_growth_pct
FROM monthly
ORDER BY month
""").fetchdf()

**Query 3 — Revenue, Profit & Margin by Category**

Each of the 8 product categories ranked by revenue, with gross profit and gross margin % calculated from the cost-to-price spread. Revenue alone can be misleading — a high-revenue category with thin margins may be worth less than a smaller, high-margin one. This query shows both so the comparison is fair.

In [ ]:
con.execute("""
SELECT p.category,
       SUM(oi.quantity)                                   AS units_sold,
       ROUND(SUM(oi.line_total), 0)                       AS revenue_eur,
       ROUND(SUM((oi.unit_price - p.unit_cost) * oi.quantity
                 * (1 - oi.discount)), 0)                 AS gross_profit_eur,
       ROUND(100.0 * SUM((oi.unit_price - p.unit_cost) * oi.quantity
                         * (1 - oi.discount))
             / SUM(oi.line_total), 1)                     AS gross_margin_pct
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders   o ON oi.order_id   = o.order_id
WHERE o.order_status IN ('Delivered', 'Shipped')
GROUP BY p.category
ORDER BY revenue_eur DESC
""").fetchdf()

**Query 4 — Top 10 Products by Revenue**

The 10 highest-revenue individual products — the "hero SKUs". `RANK()` assigns an explicit position. These are the products that most need reliable stock and premium fulfilment, because a stockout here costs the most.

In [ ]:
con.execute("""
SELECT p.product_id, p.product_name, p.category,
       SUM(oi.quantity)             AS units_sold,
       ROUND(SUM(oi.line_total), 0) AS revenue_eur,
       RANK() OVER (ORDER BY SUM(oi.line_total) DESC) AS revenue_rank
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders   o ON oi.order_id   = o.order_id
WHERE o.order_status IN ('Delivered', 'Shipped')
GROUP BY p.product_id, p.product_name, p.category
ORDER BY revenue_eur DESC
LIMIT 10
""").fetchdf()

**Query 5 — Warehouse Performance**

Revenue and order volume per warehouse, with each warehouse's share of total orders calculated using a window `SUM() OVER ()`. This shows how order volume is distributed across the 6 fulfilment hubs — useful for capacity planning and for spotting whether any hub is over- or under-loaded.

In [ ]:
con.execute("""
SELECT w.warehouse_name, w.region,
       COUNT(DISTINCT o.order_id)   AS orders_fulfilled,
       ROUND(SUM(oi.line_total), 0) AS revenue_eur,
       ROUND(AVG(oi.line_total), 2) AS avg_line_value_eur,
       ROUND(100.0 * COUNT(DISTINCT o.order_id)
             / SUM(COUNT(DISTINCT o.order_id)) OVER (), 1) AS pct_of_total_orders
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN warehouses w  ON o.warehouse_id = w.warehouse_id
WHERE o.order_status IN ('Delivered', 'Shipped')
GROUP BY w.warehouse_name, w.region
ORDER BY revenue_eur DESC
""").fetchdf()

### 4.2 Customer Analytics

These three queries shift from "what sells" to "who buys": how value is distributed across customer segments, who the highest-value individuals are, and how much revenue comes from new versus returning customers.

**Query 6 — Customer Segment Performance**

Compares the three customer segments — Standard, Premium, VIP — on customer count, order volume, total revenue, revenue per customer, and orders per customer. This reveals whether the smaller premium segments punch above their weight in revenue terms, which is the usual justification for tiered loyalty programmes.

In [ ]:
con.execute("""
SELECT c.customer_segment,
       COUNT(DISTINCT c.customer_id)                                AS customers,
       COUNT(DISTINCT o.order_id)                                   AS total_orders,
       ROUND(SUM(oi.line_total), 0)                                 AS revenue_eur,
       ROUND(SUM(oi.line_total) / COUNT(DISTINCT c.customer_id), 2) AS revenue_per_customer,
       ROUND(COUNT(DISTINCT o.order_id)::DECIMAL
             / COUNT(DISTINCT c.customer_id), 2)                    AS orders_per_customer
FROM customers c
JOIN orders      o  ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id    = oi.order_id
WHERE o.order_status IN ('Delivered', 'Shipped')
GROUP BY c.customer_segment
ORDER BY revenue_eur DESC
""").fetchdf()

**Query 7 — Top 20 Customers by Lifetime Value**

The 20 customers who have spent the most across their entire history, with their order count and the date range of their activity. These are the accounts a retention team would prioritise — losing any one of them costs more than losing a typical customer.

In [ ]:
con.execute("""
SELECT c.customer_id,
       c.first_name || ' ' || c.last_name AS customer_name,
       c.city, c.customer_segment,
       COUNT(DISTINCT o.order_id)         AS total_orders,
       ROUND(SUM(oi.line_total), 2)       AS lifetime_value_eur,
       MIN(o.order_date)                  AS first_order,
       MAX(o.order_date)                  AS last_order
FROM customers c
JOIN orders      o  ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id    = oi.order_id
WHERE o.order_status IN ('Delivered', 'Shipped')
GROUP BY c.customer_id, c.first_name, c.last_name, c.city, c.customer_segment
ORDER BY lifetime_value_eur DESC
LIMIT 20
""").fetchdf()

**Query 8 — New vs Returning Customer Revenue**

Splits revenue between first-ever orders and repeat orders. A `ROW_NUMBER()` window partitioned by customer and ordered by date labels each order as the customer's 1st, 2nd, 3rd, and so on — order number 1 is "new", everything else is "returning". The revenue split between the two is a direct read on whether the business runs on acquisition or on retention.

In [ ]:
con.execute("""
WITH order_seq AS (
    SELECT o.order_id, o.customer_id, o.order_date,
           ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date) AS order_num
    FROM orders o
    WHERE o.order_status IN ('Delivered', 'Shipped')
)
SELECT
    CASE WHEN os.order_num = 1 THEN 'New customer' ELSE 'Returning customer' END AS customer_type,
    COUNT(DISTINCT os.order_id)                                            AS orders,
    ROUND(SUM(oi.line_total), 0)                                           AS revenue_eur,
    ROUND(100.0 * SUM(oi.line_total) / SUM(SUM(oi.line_total)) OVER (), 1) AS pct_of_revenue
FROM order_seq os
JOIN order_items oi ON os.order_id = oi.order_id
GROUP BY 1
ORDER BY revenue_eur DESC
""").fetchdf()

### 4.3 Operations & Supply Chain

These five queries cover the operational side: delivery performance by carrier and warehouse, return rates, return root causes, and payment behaviour.

**Query 9 — OTIF by Carrier**

OTIF (On-Time-In-Full) is the single most important supply chain KPI. This query ranks the 5 carriers by their on-time delivery rate, alongside average delivery days and average delay. The gap between the best and worst carrier is the headline operational finding of the whole project.

In [ ]:
con.execute("""
SELECT carrier,
       COUNT(*)                                                AS total_shipments,
       SUM(on_time_delivery)                                   AS on_time_count,
       ROUND(100.0 * AVG(on_time_delivery), 2)                 AS on_time_pct,
       ROUND(AVG(actual_delivery_days), 2)                     AS avg_delivery_days,
       ROUND(AVG(actual_delivery_days - promised_delivery_days), 2) AS avg_delay_days
FROM shipments
GROUP BY carrier
ORDER BY on_time_pct DESC
""").fetchdf()

**Query 10 — Fulfilment Speed by Warehouse**

Average delivery time and on-time rate per warehouse. Joining shipments to orders to warehouses links every delivery back to the hub that fulfilled it. A slow warehouse is a bottleneck for every customer it serves, so isolating one is directly actionable.

In [ ]:
con.execute("""
SELECT w.warehouse_name, w.city,
       COUNT(s.shipment_id)                      AS shipments,
       ROUND(AVG(s.actual_delivery_days), 2)     AS avg_delivery_days,
       ROUND(100.0 * AVG(s.on_time_delivery), 2) AS on_time_pct
FROM shipments s
JOIN orders     o ON s.order_id     = o.order_id
JOIN warehouses w ON o.warehouse_id = w.warehouse_id
GROUP BY w.warehouse_name, w.city
ORDER BY avg_delivery_days ASC
""").fetchdf()

**Query 11 — Return Rate by Category**

Return rate per category — returned orders as a percentage of total orders — plus the total refund cost each category carries. Two CTEs do the counting separately (one for all orders, one for returned orders) and a `LEFT JOIN` brings them together. A high return rate is both a cost problem and a signal of a product-quality or sizing issue.

In [ ]:
con.execute("""
WITH cat_orders AS (
    SELECT p.category, COUNT(DISTINCT oi.order_id) AS total_orders
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category
),
cat_returns AS (
    SELECT p.category,
           COUNT(DISTINCT r.order_id) AS returned_orders,
           SUM(r.refund_amount)       AS refund_eur
    FROM returns r
    JOIN order_items oi ON r.order_id   = oi.order_id
    JOIN products    p  ON oi.product_id = p.product_id
    GROUP BY p.category
)
SELECT co.category, co.total_orders, cr.returned_orders,
       ROUND(100.0 * cr.returned_orders / co.total_orders, 2) AS return_rate_pct,
       ROUND(cr.refund_eur, 0)                                AS total_refunds_eur
FROM cat_orders co
LEFT JOIN cat_returns cr ON co.category = cr.category
ORDER BY return_rate_pct DESC
""").fetchdf()

**Query 12 — Top Return Reasons**

Every return reason ranked by frequency, with its share of all returns and the refund cost attached to it. This is the root-cause view — knowing *why* customers return things points directly at what to fix (for example, a dominant "Wrong size" reason justifies investing in better size guides).

In [ ]:
con.execute("""
SELECT return_reason,
       COUNT(*)                                          AS return_count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_returns,
       ROUND(SUM(refund_amount), 0)                      AS total_refund_eur,
       ROUND(AVG(refund_amount), 2)                      AS avg_refund_eur
FROM returns
GROUP BY return_reason
ORDER BY return_count DESC
""").fetchdf()

**Query 13 — Payment Method Mix**

How orders and revenue split across the 6 payment methods. Beyond simple curiosity, this informs checkout design — if a method like Klarna or PayPal carries a large share, its reliability and fees directly affect the business.

In [ ]:
con.execute("""
SELECT o.payment_method,
       COUNT(*)                                          AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_orders,
       ROUND(SUM(oi.line_total), 0)                      AS revenue_eur,
       ROUND(AVG(oi.line_total), 2)                      AS avg_line_value_eur
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status IN ('Delivered', 'Shipped')
GROUP BY o.payment_method
ORDER BY orders DESC
""").fetchdf()

### 4.4 Advanced Analytics

The final two queries apply techniques that marketing and analytics teams use directly: RFM segmentation for campaign targeting, and cohort retention for measuring whether customers keep coming back.

**Query 14 — RFM Customer Segmentation**

RFM scores every customer on three dimensions — **Recency** (how recently they ordered), **Frequency** (how often), and **Monetary** (how much they spent). `NTILE(5)` splits customers into five equal buckets on each dimension, and a `CASE` statement combines the three scores into named segments like "Champions", "At risk", and "Hibernating". Marketing teams use exactly this to decide who gets which campaign — you reward Champions and try to win back the At-risk group.

In [ ]:
con.execute("""
WITH customer_metrics AS (
    SELECT c.customer_id,
           DATE_DIFF('day', MAX(o.order_date), DATE '2024-12-31') AS recency_days,
           COUNT(DISTINCT o.order_id)                             AS frequency,
           SUM(oi.line_total)                                     AS monetary
    FROM customers c
    JOIN orders      o  ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id    = oi.order_id
    WHERE o.order_status IN ('Delivered', 'Shipped')
    GROUP BY c.customer_id
),
rfm_scores AS (
    SELECT customer_id, recency_days, frequency, monetary,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
           NTILE(5) OVER (ORDER BY frequency ASC)    AS f_score,
           NTILE(5) OVER (ORDER BY monetary ASC)     AS m_score
    FROM customer_metrics
)
SELECT
    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN '01 Champions'
        WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN '02 Loyal customers'
        WHEN r_score >= 4 AND f_score <= 2                  THEN '03 New customers'
        WHEN r_score <= 2 AND f_score >= 3                  THEN '04 At risk'
        WHEN r_score <= 2 AND f_score <= 2 AND m_score >= 3 THEN '05 Cant lose them'
        WHEN r_score <= 2 AND f_score <= 2                  THEN '06 Hibernating'
        ELSE                                                     '07 Others'
    END                              AS rfm_segment,
    COUNT(*)                         AS customer_count,
    ROUND(AVG(monetary), 0)          AS avg_lifetime_value_eur,
    ROUND(AVG(recency_days), 0)      AS avg_days_since_last_order,
    ROUND(AVG(frequency), 1)         AS avg_orders
FROM rfm_scores
GROUP BY rfm_segment
ORDER BY rfm_segment
""").fetchdf()

**Query 15 — Monthly Cohort Retention**

Cohort retention groups customers by the month of their first order, then tracks what percentage of each cohort comes back to order again 1, 3, 6, and 12 months later. The `DATE_DIFF` between the cohort month and each subsequent order date gives the "month offset", and conditional `COUNT(DISTINCT ...)` counts how many of the original cohort were still active at each offset. This is the metric subscription and e-commerce businesses live by — it answers "do customers stick around?" rather than just "did we make a sale?".

In [ ]:
con.execute("""
WITH first_order AS (
    SELECT customer_id, DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders
    WHERE order_status IN ('Delivered', 'Shipped')
    GROUP BY customer_id
),
customer_activity AS (
    SELECT fo.cohort_month, o.customer_id,
           DATE_DIFF('month', fo.cohort_month, DATE_TRUNC('month', o.order_date)) AS month_offset
    FROM first_order fo
    JOIN orders o ON fo.customer_id = o.customer_id
    WHERE o.order_status IN ('Delivered', 'Shipped')
)
SELECT cohort_month,
       COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END) AS cohort_size,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN month_offset = 1 THEN customer_id END)
             / NULLIF(COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END), 0), 1) AS month_1_pct,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN month_offset = 3 THEN customer_id END)
             / NULLIF(COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END), 0), 1) AS month_3_pct,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN month_offset = 6 THEN customer_id END)
             / NULLIF(COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END), 0), 1) AS month_6_pct,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN month_offset = 12 THEN customer_id END)
             / NULLIF(COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END), 0), 1) AS month_12_pct
FROM customer_activity
GROUP BY cohort_month
HAVING COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END) > 100
ORDER BY cohort_month
""").fetchdf()

## 5. Export Results for Tableau

This cell re-runs the key queries and saves their results as small CSV files in a `sql_results/` folder. These aggregated files become the data sources for the Tableau dashboards — they are tiny compared to the raw 3.4M-row tables, so Tableau loads and refreshes them instantly.

In [ ]:
import os

RESULTS_PATH = "/content/drive/MyDrive/BlitzMart_Analytics/sql_results"
os.makedirs(RESULTS_PATH, exist_ok=True)

queries = {
    "q01_topline_kpis": """
        SELECT COUNT(DISTINCT o.order_id) AS total_orders,
               COUNT(DISTINCT o.customer_id) AS unique_customers,
               ROUND(SUM(oi.line_total), 0) AS total_revenue_eur,
               ROUND(SUM(oi.line_total) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value_eur
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status IN ('Delivered', 'Shipped')""",
    "q02_monthly_revenue": """
        SELECT DATE_TRUNC('month', o.order_date) AS month,
               COUNT(DISTINCT o.order_id) AS orders,
               ROUND(SUM(oi.line_total), 0) AS revenue_eur
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status IN ('Delivered', 'Shipped')
        GROUP BY 1 ORDER BY 1""",
    "q03_category_perf": """
        SELECT p.category, ROUND(SUM(oi.line_total), 0) AS revenue_eur,
               ROUND(SUM((oi.unit_price - p.unit_cost) * oi.quantity * (1 - oi.discount)), 0) AS gross_profit_eur
        FROM order_items oi JOIN products p ON oi.product_id = p.product_id
        JOIN orders o ON oi.order_id = o.order_id
        WHERE o.order_status IN ('Delivered', 'Shipped')
        GROUP BY p.category ORDER BY revenue_eur DESC""",
    "q09_carrier_otif": """
        SELECT carrier, COUNT(*) AS shipments,
               ROUND(100.0 * AVG(on_time_delivery), 2) AS on_time_pct,
               ROUND(AVG(actual_delivery_days), 2) AS avg_delivery_days
        FROM shipments GROUP BY carrier ORDER BY on_time_pct DESC""",
    "q11_return_rate": """
        WITH co AS (SELECT p.category, COUNT(DISTINCT oi.order_id) AS total_orders
                    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
                    GROUP BY p.category),
             cr AS (SELECT p.category, COUNT(DISTINCT r.order_id) AS returned_orders
                    FROM returns r JOIN order_items oi ON r.order_id = oi.order_id
                    JOIN products p ON oi.product_id = p.product_id GROUP BY p.category)
        SELECT co.category, co.total_orders, cr.returned_orders,
               ROUND(100.0 * cr.returned_orders / co.total_orders, 2) AS return_rate_pct
        FROM co LEFT JOIN cr ON co.category = cr.category ORDER BY return_rate_pct DESC""",
}

for name, sql in queries.items():
    df = con.execute(sql).fetchdf()
    df.to_csv(f"{RESULTS_PATH}/{name}.csv", index=False)
    print(f"  {name}.csv  ({len(df)} rows)")

print(f"\nSaved to: {RESULTS_PATH}")

---

## Summary

This notebook took the cleaned dataset through 15 SQL queries spanning revenue, customers, operations, and advanced analytics. The results feed directly into Notebook 3 (Python analysis) and the Tableau dashboards.

The key techniques demonstrated: CTEs for breaking complex logic into readable steps, window functions (`LAG`, `RANK`, `ROW_NUMBER`, `NTILE`, `SUM() OVER ()`) for period-over-period and share-of-total calculations, RFM segmentation for customer targeting, and cohort analysis for retention measurement — all on a 6.6M-row dataset.